In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [ ]:
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle
!kaggle datasets download -d nih-chest-xrays/sample
!unzip -q sample.zip -d /content/nih_sample

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/sample
License(s): CC0-1.0
100% 4.20G/4.20G [00:35<00:00, 126MB/s]



In [ ]:
for root, dirs, files in os.walk('/content/nih_sample'):
    print(root, '->', len(files), 'files')
    if len(files) > 0:
        print('   example:', files[0])

/content/nih_sample -> 1 files
   example: sample_labels.csv
/content/nih_sample/sample -> 1 files
   example: sample_labels.csv
/content/nih_sample/sample/sample -> 0 files
/content/nih_sample/sample/sample/images -> 5606 files
   example: 00015031_031.png
/content/nih_sample/sample/images -> 5606 files
   example: 00015031_031.png


In [ ]:
import os

dir1 = '/content/nih_sample/sample/images'
dir2 = '/content/nih_sample/sample/sample/images'

files1 = set(os.listdir(dir1))
files2 = set(os.listdir(dir2))

print(files1 == files2)

True


In [ ]:
import shutil
shutil.rmtree('/content/nih_sample/sample/sample')

In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   54G   60G  48% /


In [ ]:
csv_path = '/content/nih_sample/sample/sample_labels.csv'
image_dir = '/content/nih_sample/sample/images'

import pandas as pd
df = pd.read_csv(csv_path)
print(df.shape)
print(df.columns.tolist())
df.head()

(5606, 11)
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImageWidth', 'OriginalImageHeight', 'OriginalImagePixelSpacing_x', 'OriginalImagePixelSpacing_y']


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImageWidth,OriginalImageHeight,OriginalImagePixelSpacing_x,OriginalImagePixelSpacing_y
0,00000013_005.png,Emphysema|Infiltration|Pleural_Thickening|Pneu...,5,13,060Y,M,AP,3056,2544,0.139,0.139
1,00000013_026.png,Cardiomegaly|Emphysema,26,13,057Y,M,AP,2500,2048,0.168,0.168
2,00000017_001.png,No Finding,1,17,077Y,M,AP,2500,2048,0.168,0.168
3,00000030_001.png,Atelectasis,1,30,079Y,M,PA,2992,2991,0.143,0.143
4,00000032_001.png,Cardiomegaly|Edema|Effusion,1,32,055Y,F,AP,2500,2048,0.168,0.168


In [ ]:
all_labels = df['Finding Labels'].str.split('|').explode().unique()
print(sorted(all_labels))

['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


DATA PREPROCESSING

In [ ]:
PATHOLOGIES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

def encode_labels(finding_str):
    labels = finding_str.split('|')
    return [1 if p in labels else 0 for p in PATHOLOGIES]

label_matrix = df['Finding Labels'].apply(encode_labels)
label_df = pd.DataFrame(label_matrix.tolist(), columns=PATHOLOGIES)
df = pd.concat([df, label_df], axis=1)

print(df[PATHOLOGIES].sum().sort_values(ascending=False))

Infiltration          967
Effusion              644
Atelectasis           508
Nodule                313
Mass                  284
Pneumothorax          271
Consolidation         226
Pleural_Thickening    176
Cardiomegaly          141
Emphysema             127
Edema                 118
Fibrosis               84
Pneumonia              62
Hernia                 13
dtype: int64


TRAIN TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

unique_patients = df['Patient ID'].unique()
train_ids, val_ids = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_df = df[df['Patient ID'].isin(train_ids)].reset_index(drop=True)
val_df = df[df['Patient ID'].isin(val_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} images, {len(train_ids)} patients")
print(f"Val: {len(val_df)} images, {len(val_ids)} patients")

Train: 4462 images, 3384 patients
Val: 1144 images, 846 patients


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.image_dir}/{row['Image Index']}"
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(row[PATHOLOGIES].values.astype('float32'))
        return image, labels

TRANSFORM

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1,contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

MAKING BATCHES

In [ ]:
from torch.utils.data import DataLoader

train_dataset = ChestXrayDataset(train_df, image_dir, transform=train_transform)
val_dataset = ChestXrayDataset(val_df, image_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32, 14])


In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.densenet121(weights='IMAGENET1K_V1')
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, len(PATHOLOGIES))
model = model.to(device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 175MB/s]


In [ ]:
pos_counts = train_df[PATHOLOGIES].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

In [ ]:
print(model)

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

model.eval()
with torch.no_grad():
    sample_images, _ = next(iter(train_loader))
    sample_output = model(sample_images.to(device))
    print(sample_output.shape)

Total parameters: 6,968,206
Trainable parameters: 2,172,430
torch.Size([32, 14])


In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters now: {trainable_params:,}")

Trainable parameters now: 14,350


In [ ]:
for param in model.features.denseblock4.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters now: {trainable_params:,}")

Trainable parameters now: 2,172,430


In [ ]:
pos_counts = train_df[PATHOLOGIES].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

train_losses = []
val_losses = []
mean_aurocs = []
per_epoch_aurocs = []

best_auroc = 0
patience = 5
patience_counter = 0
num_epochs = 25

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    aurocs = []
    for i, pathology in enumerate(PATHOLOGIES):
        if len(np.unique(all_labels[:, i])) > 1:
            auroc = roc_auc_score(all_labels[:, i], all_preds[:, i])
        else:
            auroc = float('nan')
        aurocs.append(auroc)

    mean_auroc = np.nanmean(aurocs)
    scheduler.step(mean_auroc)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    mean_aurocs.append(mean_auroc)
    per_epoch_aurocs.append(aurocs)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Mean AUROC: {mean_auroc:.4f}")

    if mean_auroc > best_auroc:
        best_auroc = mean_auroc
        patience_counter = 0
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model.pth')
        print(f"  -> New best model saved (AUROC: {mean_auroc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  -> Stopping early at epoch {epoch+1}, no improvement for {patience} epochs")
            break

Epoch 1/25 | Train Loss: 1.1254 | Val Loss: 1.3021 | Mean AUROC: 0.6992
  -> New best model saved (AUROC: 0.6992)
Epoch 2/25 | Train Loss: 0.9910 | Val Loss: 1.3758 | Mean AUROC: 0.7010
  -> New best model saved (AUROC: 0.7010)
Epoch 3/25 | Train Loss: 0.9315 | Val Loss: 1.3622 | Mean AUROC: 0.7071
  -> New best model saved (AUROC: 0.7071)
Epoch 4/25 | Train Loss: 0.8642 | Val Loss: 1.3495 | Mean AUROC: 0.7207
  -> New best model saved (AUROC: 0.7207)
Epoch 5/25 | Train Loss: 0.8338 | Val Loss: 1.3358 | Mean AUROC: 0.7200
Epoch 6/25 | Train Loss: 0.8081 | Val Loss: 1.3204 | Mean AUROC: 0.7252
  -> New best model saved (AUROC: 0.7252)
Epoch 7/25 | Train Loss: 0.7614 | Val Loss: 1.3784 | Mean AUROC: 0.7308
  -> New best model saved (AUROC: 0.7308)
Epoch 8/25 | Train Loss: 0.7443 | Val Loss: 1.3768 | Mean AUROC: 0.7302
Epoch 9/25 | Train Loss: 0.6987 | Val Loss: 1.4109 | Mean AUROC: 0.7313
  -> New best model saved (AUROC: 0.7313)
Epoch 10/25 | Train Loss: 0.6741 | Val Loss: 1.5325 | Mean

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

torch.save(model, '/content/drive/MyDrive/chexray_densenet121_full2.pth')
print("Full model saved to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full model saved to Google Drive.
